[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [LightGBM and XGBoost, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)

# Missing Values


## What you will be able to do

Train LightGBM and XGBoost on a table with gaps in it, read where each split sends a missing value,
and catch the gaps that a model was never trained to handle before they reach it.

## The idea

### The problem

Thirty percent of this guide's made-up stations have no snow sensor, so `snow_cm` is `NaN` on every
one of their days, which in the 15,000 training rows makes 4,750 gaps, almost a third of the column. A
model that needs every cell filled leaves two choices. Dropping the rows with a gap throws away 95 of
the 300 training stations, and everything else their days record. Filling each gap with a number makes
it look like a reading. The training median is 0.0 cm, because 57% of the readings are bare ground, so
a median fill tells the model that every station without a sensor had bare ground on every day.

The **Why Gradient Boosting** notebook filled the gaps with the median for its linear model, which
cannot take a missing value, and handed the column to both boosting libraries as it was. This
notebook looks at what the libraries do with it.

### A formal definition

> In LightGBM and XGBoost, a **missing value** is a `NaN` in a feature column. On a feature that had
> gaps in training, each split that rows with a gap reach learns a **default direction**, the side
> those rows are sent to. Both sides are tried, and the side that gains more keeps them. A split that
> no row with a gap reaches is given a default direction without that test. A new row with a gap
> follows the default direction at every split it reaches.

### Why it works that way

A split sorts the known values into two groups at a threshold, and a gap has no place on that line.
Trying both sides costs one more sum for each candidate split, and it lets the trees decide, split by
split, whether the days without a reading behave like the snowy days or the bare ones. Different
splits can decide differently, so a gap is not one number in disguise, as a fill would make it.

That choice can only be learned where the training rows had gaps. A gap that first appears in new
data, in a column that was complete in training, is still sent somewhere, and without a warning.
LightGBM reads it as 0, and XGBoost sends it along a default direction that no training row tested.
A gap in the answers is another matter, since a row with no answer has nothing to teach. XGBoost
refuses one. LightGBM checks the answers only when the features arrive as a NumPy array, and given a
DataFrame, as here, it trains on a missing answer as if it were 0.

### Where this shows up

Tables gathered from instruments or forms have gaps, from a sensor never installed, a reading lost
in transit or a question left blank. The table every notebook in this
guide starts from has its gaps in `snow_cm`.

### What this notebook covers

- the gaps in the snow column, and what a fill would put in them
- where one split sends the gaps, checked by hand
- the gaps left in, against a fill, a declared marker and no snow column at all
- four mistakes, two of which raise nothing

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import numpy as np
import xgboost as xgb

x = np.array([[1.0], [2.0], [3.0], [8.0], [9.0], [np.nan], [np.nan]])
new = np.array([[2.0], [9.0], [np.nan]])
for gaps in ([9.0, 11.0], [-1.0, 1.0]):
    y = np.array([0.0, 0.0, 0.0, 10.0, 10.0] + gaps)
    stump = xgb.XGBRegressor(n_estimators=1, max_depth=1, learning_rate=1.0, reg_lambda=0,
                             base_score=0).fit(x, y)
    print(gaps, stump.predict(new))
```

```
[9.0, 11.0] [ 0. 10. 10.]
[-1.0, 1.0] [ 0. 10.  0.]
```

Each model is one split between 3 and 8, with its full step added. When the two rows with a gap have
answers near 10, the split sends gaps to the right, and a new row with a gap is predicted 10. When
their answers are near 0, the gaps go left, and the same new row is predicted 0.

## Setup

The table, the two boosting libraries, and a score.

- `numpy` and `pandas` build the table
- `lightgbm` and `xgboost` are LightGBM and XGBoost, and the cell prints their versions
- `mean_squared_error` scores a model's predictions

`station_days` makes the same 20,000 made-up station-days as the **Why Gradient Boosting** notebook,
drawn from a generator started from a fixed seed, so the table is the same on every run. The cell
splits the stations into 300 for training, 50 for validation and 50 for testing, keeping all of a
station's days on one side, and then takes each part's eight features as `X_train`, `X_valid` and
`X_test` and its lowest temperatures as `y_train`, `y_valid` and `y_test`. Common errors imports `re`
and `warnings` where it needs them.

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import mean_squared_error

TERRAINS = ["coastal", "fjord", "inland", "valley", "mountain"]


def station_days(n_stations=400, days=50, seed=2026):
    """Made-up daily minimum temperatures at weather stations in Norway, one row per station and day."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n_stations)
    coast_km = np.minimum(rng.exponential(30, n_stations), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n_stations) + 2.5 * coast_km, 0, 1600)
    terrain = rng.choice(TERRAINS, n_stations, p=[0.2, 0.2, 0.25, 0.2, 0.15])
    has_snow_sensor = rng.uniform(size=n_stations) < 0.7
    offset = rng.normal(0, 0.8, n_stations)
    station = np.repeat(np.arange(n_stations), days)
    n = n_stations * days
    day = rng.integers(1, 366, n)
    cloud = np.clip(rng.beta(1.2, 1.0, n), 0, 1)
    wind = rng.gamma(2.0, 2.0, n)
    season = np.cos(2 * np.pi * (day - 20) / 365)
    lat, elev, coast = latitude[station], elevation[station], coast_km[station]
    snow = np.where(season > 0.2, np.clip(rng.normal(30 * season + 0.03 * elev + 2 * (lat - 58), 15), 0, None), 0.0)
    shelter = pd.Series(terrain[station]).map({"coastal": 0.4, "fjord": 1.0, "inland": 1.2, "valley": 1.7, "mountain": 0.8}).to_numpy()
    mean = 8 - 0.55 * (lat - 58) - 0.006 * elev - 9 * season + 3.5 * np.exp(-coast / 40) * season
    cooling = 7 * (1 - cloud) * np.exp(-wind / 3) * (0.4 + (season + 1) / 2) * shelter
    tmin = mean - 2.5 - cooling - 2.0 * (snow > 10) * (1 - cloud) + offset[station] + rng.normal(0, 1.2, n)
    snow_cm = np.where(has_snow_sensor[station], snow.round(0), np.nan)
    return pd.DataFrame({
        "station": [f"ST{s + 1:04d}" for s in station],
        "latitude": lat.round(2), "elevation_m": elev.round(0).astype(int), "coast_km": coast.round(1),
        "terrain": pd.Categorical(terrain[station], categories=TERRAINS),
        "day_of_year": day, "cloud_cover": cloud.round(2), "wind_ms": wind.round(1), "snow_cm": snow_cm,
        "tmin": tmin.round(1),
    })


FEATURES = ["latitude", "elevation_m", "coast_km", "terrain", "day_of_year", "cloud_cover", "wind_ms", "snow_cm"]
days = station_days()
stations = days["station"].unique()
order = np.random.default_rng(0).permutation(len(stations))
train = days[days["station"].isin(stations[order[:300]])]
valid = days[days["station"].isin(stations[order[300:350]])]
test = days[days["station"].isin(stations[order[350:]])]
X_train, y_train = train[FEATURES], train["tmin"]
X_valid, y_valid = valid[FEATURES], valid["tmin"]
X_test, y_test = test[FEATURES], test["tmin"]

print("lightgbm", lgb.__version__, "| xgboost", xgb.__version__, "|", len(train), "training rows,",
      len(valid), "validation rows,", len(test), "test rows")

lightgbm 4.6.0 | xgboost 3.4.1 | 15000 training rows, 2500 validation rows, 2500 test rows


## Worked examples

### The gaps in the snow column

A station either has a snow sensor or it does not, so its gaps come on all of its days at once.

In [2]:
for name, part in (("training", train), ("validation", valid), ("test", test)):
    no_sensor = part.groupby("station")["snow_cm"].apply(lambda readings: readings.isna().all())
    print(f"{name:10}: {part['snow_cm'].isna().mean():.3f} of the rows have a gap, "
          f"{no_sensor.sum()} of {len(no_sensor)} stations have no sensor")

share = train.groupby("station")["snow_cm"].apply(lambda readings: readings.isna().mean())
print("every training station has gaps on all of its days or on none:", share.isin([0, 1]).all())

training  : 0.317 of the rows have a gap, 95 of 300 stations have no sensor
validation: 0.320 of the rows have a gap, 16 of 50 stations have no sensor
test      : 0.300 of the rows have a gap, 15 of 50 stations have no sensor
every training station has gaps on all of its days or on none: True


About three rows in ten have a gap in each part, from 95 of the 300 training stations, 16 of the 50
validation stations and 15 of the 50 test stations.

### What a filled gap would say

In [3]:
readings = X_train["snow_cm"].dropna()
median = readings.median()
print("training median:", median, "| share of the readings at 0 cm:", round((readings == 0).mean(), 3))

filled = X_train["snow_cm"].fillna(median)
print("rows reading 0 cm before the fill:", (X_train["snow_cm"] == 0).sum(), "| after it:", (filled == 0).sum())

training median: 0.0 | share of the readings at 0 cm: 0.571
rows reading 0 cm before the fill: 5853 | after it: 10603


More than half of the measured days had no snow, so the median is 0.0. After a median fill, 10,603
training rows read 0 cm, and nothing in the column tells the 5,853 days of bare ground from the 4,750
days at stations with no sensor.

### Training with the gaps left in

The helpers below train a model on the tables they are given, stop it early on the validation
stations at a learning rate of 0.1, and return the model with its number of trees and its validation
loss. Neither library needs anything done to the gaps.

In [4]:
def lightgbm_validation(train_X, valid_X, **settings):
    model = lgb.LGBMRegressor(n_estimators=3000, learning_rate=0.1, random_state=0, verbose=-1, **settings)
    model.fit(train_X, y_train, eval_set=[(valid_X, y_valid)], callbacks=[lgb.early_stopping(100, verbose=False)])
    return model, model.best_iteration_, round(float(model.best_score_["valid_0"]["l2"]), 3)


def xgboost_validation(train_X, valid_X, **settings):
    model = xgb.XGBRegressor(n_estimators=3000, learning_rate=0.1, early_stopping_rounds=100, random_state=0,
                             **settings)
    model.fit(train_X, y_train, eval_set=[(valid_X, y_valid)], verbose=False)
    return model, model.best_iteration + 1, round(mean_squared_error(y_valid, model.predict(valid_X)), 3)


lightgbm_model, trees, loss = lightgbm_validation(X_train, X_valid)
print(f"LightGBM: {trees} trees, validation loss {loss}")
xgboost_model, trees, loss = xgboost_validation(X_train, X_valid)
print(f"XGBoost:  {trees} trees, validation loss {loss}")

LightGBM: 120 trees, validation loss 2.546
XGBoost:  77 trees, validation loss 2.526


LightGBM stops after 120 trees at 2.546 and XGBoost after 77 at 2.526, the same models as the default
settings gave in the **Regularization and Sampling** notebook, gaps and all.

### Where one split sends the gaps

The cell below fits a single XGBoost split on `snow_cm` alone, with its full step, no penalty and a
starting prediction of 0, so that its choice can be checked by hand. In XGBoost's table of nodes,
`Yes` is the side for values below `Split`, `No` the side for the rest, and `Missing` the side the
gaps take.

In [5]:
stump = xgb.XGBRegressor(n_estimators=1, max_depth=1, learning_rate=1.0, reg_lambda=0, base_score=0)
stump.fit(X_train[["snow_cm"]], y_train)
nodes = stump.get_booster().trees_to_dataframe()
print(nodes[["ID", "Feature", "Split", "Yes", "No", "Missing", "Cover"]])

    ID  Feature  Split   Yes    No Missing    Cover
0  0-0  snow_cm   11.0   0-1   0-2     0-1  15000.0
1  0-1     Leaf   <NA>  <NA>  <NA>    <NA>  10735.0
2  0-2     Leaf   <NA>  <NA>  <NA>    <NA>   4265.0


The split is at 11 cm, and `Missing` names `0-1`, the same node as `Yes`, so the gaps join the days
with less than 11 cm of snow. The leaf `0-1` holds 10,735 rows, which is those days and the gaps
together.

In [6]:
snow, split = X_train["snow_cm"], nodes.loc[0, "Split"]
below, above, gap = snow < split, snow >= split, snow.isna()


def gain(left, right):
    before = y_train.sum() ** 2 / len(y_train)
    return y_train[left].sum() ** 2 / left.sum() + y_train[right].sum() ** 2 / right.sum() - before


print("rows below, above and with a gap:", below.sum(), above.sum(), gap.sum())
print("their mean answers:", round(y_train[below].mean(), 3), round(y_train[above].mean(), 3),
      round(y_train[gap].mean(), 3))
print(f"gain with the gaps below: {gain(below | gap, above):,.0f}")
print(f"gain with the gaps above: {gain(below, above | gap):,.0f}")
print(f"XGBoost's gain for its split: {nodes.loc[0, 'Gain']:,.0f}")

rows below, above and with a gap: 5985 4265 4750
their mean answers: 2.919 -7.642 -1.624
gain with the gaps below: 223,180
gain with the gaps above: 196,431
XGBoost's gain for its split: 223,180


With no penalty, the gain of a split for squared error is each side's sum squared over its rows, added
up, minus the same for all the rows together. The days with a gap average −1.624 °C, nearer the
2.919 of the days with little snow than the −7.642 of the snowy ones, and sending them below gains
223,180 against 196,431. XGBoost's own gain for its split matches the first to the nearest whole
number.

In [7]:
lightgbm_stump = lgb.LGBMRegressor(n_estimators=1, num_leaves=2, learning_rate=1.0, verbose=-1)
lightgbm_stump.fit(X_train[["snow_cm"]], y_train)
lightgbm_nodes = lightgbm_stump.booster_.trees_to_dataframe()
print(lightgbm_nodes[["node_index", "threshold", "decision_type", "missing_direction", "missing_type", "count"]])

  node_index  threshold decision_type missing_direction missing_type  count
0       0-S0       10.5            <=              left          NaN  15000
1       0-L0        NaN          None              None         None  10735
2       0-L1        NaN          None              None         None   4265


LightGBM puts its threshold halfway between two readings, at `<= 10.5`, which divides whole
centimeters the way XGBoost's `< 11` does, and its `missing_direction` sends the gaps left, to the
same 10,735 rows. Its `missing_type` of `NaN` says that `snow_cm` had gaps in training, so a gap
follows `missing_direction` at this split instead of being read as 0.

### The directions across the full models

The full models from the third worked example make many splits on `snow_cm`, and each one has its
own default direction.

In [8]:
lightgbm_splits = lightgbm_model.booster_.trees_to_dataframe()
on_snow = lightgbm_splits[lightgbm_splits["split_feature"] == "snow_cm"]
print("LightGBM:", len(on_snow), "splits on snow_cm, gaps sent", on_snow["missing_direction"].value_counts().to_dict())

xgboost_splits = xgboost_model.get_booster().trees_to_dataframe()
on_snow = xgboost_splits[xgboost_splits["Feature"] == "snow_cm"]
print("XGBoost: ", len(on_snow), "splits on snow_cm, gaps sent to Yes", (on_snow["Missing"] == on_snow["Yes"]).sum(),
      "times and to No", (on_snow["Missing"] == on_snow["No"]).sum(), "times")

LightGBM: 201 splits on snow_cm, gaps sent {'right': 116, 'left': 85}
XGBoost:  727 splits on snow_cm, gaps sent to Yes 260 times and to No 467 times


LightGBM's booster keeps only its best 120 trees, and they send the gaps right at 116 of their 201
snow splits and left at 85. XGBoost's booster holds all its rounds, the 100 after its best included,
and its 727 snow splits send the gaps to the side below the threshold 260 times and to the side above
it 467 times. A fill would settle each of these by where its number falls against the threshold. A
library weighs the rows with a gap only at a split those rows reach, and a split on snow under an
earlier one that sent every gap the other way has none of them, so its direction is the library's
default.

The cell below sends the training rows with a gap through every tree, with `pred_leaf=True`, which
returns the leaf each row lands in, and walks up from those leaves to find the splits the rows pass
on the way.

In [9]:
gap_rows = X_train[X_train["snow_cm"].isna()]


def on_the_way(parent, leaves):
    found = set()
    for node in leaves:
        while node is not None and node not in found:
            found.add(node)
            node = parent.get(node)
    return found


parent = {node: up for node, up in zip(lightgbm_splits["node_index"], lightgbm_splits["parent_index"]) if isinstance(up, str)}
leaves = lightgbm_model.booster_.predict(gap_rows, pred_leaf=True)
reached = on_the_way(parent, {f"{tree}-L{leaf}" for tree, column in enumerate(leaves.T) for leaf in set(column)})
on_snow = lightgbm_splits[lightgbm_splits["split_feature"] == "snow_cm"]
hit = on_snow["node_index"].isin(reached)
print("LightGBM:", hit.sum(), "of", len(on_snow), "snow splits reached, sending gaps",
      on_snow[hit]["missing_direction"].value_counts().to_dict(), "| the others",
      on_snow[~hit]["missing_direction"].value_counts().to_dict())

parent = {}
for node, yes, no in xgboost_splits.loc[xgboost_splits["Feature"] != "Leaf", ["ID", "Yes", "No"]].itertuples(index=False):
    parent[yes] = node
    parent[no] = node
booster = xgboost_model.get_booster()
leaves = booster.predict(xgb.DMatrix(gap_rows, enable_categorical=True), pred_leaf=True,
                         iteration_range=(0, booster.num_boosted_rounds())).astype(int)
reached = on_the_way(parent, {f"{tree}-{leaf}" for tree, column in enumerate(leaves.T) for leaf in set(column)})
on_snow = xgboost_splits[xgboost_splits["Feature"] == "snow_cm"]
hit = on_snow["ID"].isin(reached).to_numpy()
side = pd.Series(np.where(on_snow["Missing"] == on_snow["Yes"], "Yes", "No"))
print("XGBoost: ", hit.sum(), "of", len(on_snow), "snow splits reached, sending gaps", side[hit].value_counts().to_dict(),
      "| the others", side[~hit].value_counts().to_dict())

LightGBM: 181 of 201 snow splits reached, sending gaps {'right': 115, 'left': 66} | the others {'left': 19, 'right': 1}
XGBoost:  532 of 727 snow splits reached, sending gaps {'No': 272, 'Yes': 260} | the others {'No': 195}


The rows with a gap reach 181 of LightGBM's 201 snow splits, which send them right 115 times and left
66 times, and of the 20 they never reach, 19 send a gap left. In XGBoost they reach 532 of the 727,
which send them to `No` 272 times and to `Yes` 260 times, and the 195 they never reach all send a gap
to `No`, the side above the threshold.

### Filling the gaps instead

The cell below runs the same helpers on four versions of the tables: the gaps left in, the gaps
filled with the training median, the gaps filled with −999, a value far below any reading, and no snow
column at all.

In [10]:
choices = {
    "gaps left in": lambda X: X,
    "median fill": lambda X: X.fillna({"snow_cm": median}),
    "-999 fill": lambda X: X.fillna({"snow_cm": -999.0}),
    "no snow column": lambda X: X.drop(columns="snow_cm"),
}
for name, prepare in choices.items():
    _, lightgbm_trees, lightgbm_loss = lightgbm_validation(prepare(X_train), prepare(X_valid))
    _, xgboost_trees, xgboost_loss = xgboost_validation(prepare(X_train), prepare(X_valid))
    print(f"{name:14}: LightGBM {lightgbm_trees:3} trees, {lightgbm_loss} | "
          f"XGBoost {xgboost_trees:3} trees, {xgboost_loss}")

gaps left in  : LightGBM 120 trees, 2.546 | XGBoost  77 trees, 2.526
median fill   : LightGBM  87 trees, 2.574 | XGBoost  82 trees, 2.556
-999 fill     : LightGBM 148 trees, 2.559 | XGBoost 237 trees, 2.526
no snow column: LightGBM 166 trees, 2.504 | XGBoost 194 trees, 2.524


On these validation stations, in one run, the four choices end within 0.07 of each other for LightGBM
and 0.04 for XGBoost. The median fill is the worst of the four for both libraries, at 2.574 and 2.556.
A fill of −999 puts the gaps below every reading, where one threshold can separate them, and it lands
within 0.02 of the gaps left in. Dropping the column does at least as well as keeping it on these
stations, 2.504 and 2.524. Every difference in the table is a few hundredths, and one choice of
validation stations cannot rank differences that small, as the **Tuning** notebook shows with
cross-validation. Leaving the gaps in needs no decision about what to put in them.

### Declaring a marker as missing

A table that arrives with −999 where a reading is missing can be given to XGBoost as it is, with
`missing=-999` telling it which value is a gap.

In [11]:
marked = lambda X: X.fillna({"snow_cm": -999.0})
settings = dict(n_estimators=100, learning_rate=0.1, random_state=0)

with_gaps = xgb.XGBRegressor(**settings).fit(X_train, y_train)
declared = xgb.XGBRegressor(**settings, missing=-999.0).fit(marked(X_train), y_train)
undeclared = xgb.XGBRegressor(**settings).fit(marked(X_train), y_train)

print("missing=-999, same predictions as the gaps:", np.array_equal(declared.predict(marked(X_valid)), with_gaps.predict(X_valid)))
print("-999 as a reading, same predictions:      ", np.array_equal(undeclared.predict(marked(X_valid)), with_gaps.predict(X_valid)))

missing=-999, same predictions as the gaps: True
-999 as a reading, same predictions:       False


Declared, the marker gives the same predictions as the gaps themselves. Left undeclared, −999 is a
reading of minus ten meters of snow, and the trees treat it as one, which gives other predictions.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/08-missing-values-solutions.ipynb).

**1.** Print the share of the test rows with a gap in `snow_cm`, and the number of test stations with no snow sensor.

In [12]:
# your code here

**2.** Fit LightGBM's stump on `snow_cm` alone, as in the worked examples, and print the mean answer on each side of its split, with the gaps counted on the side its `missing_direction` names.

In [13]:
# your code here

**3.** Add a column `has_sensor`, 1 where the snow was measured and 0 where it was not, fill the gaps with the training median, and print both libraries' trees and validation losses with the worked examples' helpers.

In [14]:
# your code here

**4.** Make a copy of the training rows with a gap in `cloud_cover` on 10% of them, chosen with `np.random.default_rng(0)`, train LightGBM on the copy, and print the missing types of its splits on `cloud_cover`.

In [15]:
# your code here

**5.** Make a copy of the validation stations with a gap in `snow_cm` on every row, and print each library's validation loss on it beside its loss on the real readings.

In [16]:
# your code here

## Common errors

### No error, and a missing wind that LightGBM reads as calm and XGBoost as a gale

Every training row has a wind reading, and the new data has a gap in every one.

In [17]:
windless_lightgbm = lgb.LGBMRegressor(n_estimators=100, learning_rate=0.1, random_state=0, verbose=-1).fit(X_train, y_train)
windless_xgboost = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=0).fit(X_train, y_train)

no_wind = X_valid.assign(wind_ms=np.nan)
calm = X_valid.assign(wind_ms=0.0)
gale = X_valid.assign(wind_ms=X_train["wind_ms"].max())
print("LightGBM, a gap predicts the same as 0 m/s:",
      np.array_equal(windless_lightgbm.predict(no_wind), windless_lightgbm.predict(calm)))
print("XGBoost, a gap predicts the same as", X_train["wind_ms"].max(), "m/s:",
      np.array_equal(windless_xgboost.predict(no_wind), windless_xgboost.predict(gale)))

for name, model in (("LightGBM", windless_lightgbm), ("XGBoost", windless_xgboost)):
    print(f"{name}: validation loss {mean_squared_error(y_valid, model.predict(X_valid)):.3f} with the wind, "
          f"{mean_squared_error(y_valid, model.predict(no_wind)):.3f} without it | mean prediction "
          f"{model.predict(X_valid).mean():.3f} with the wind, {model.predict(no_wind).mean():.3f} without it")

LightGBM, a gap predicts the same as 0 m/s: True
XGBoost, a gap predicts the same as 26.5 m/s: True
LightGBM: validation loss 2.553 with the wind, 7.788 without it | mean prediction -0.986 with the wind, -2.574 without it
XGBoost: validation loss 2.544 with the wind, 5.521 without it | mean prediction -0.953 with the wind, -0.052 without it


In [18]:
windless_lightgbm_splits = windless_lightgbm.booster_.trees_to_dataframe()
on_wind = windless_lightgbm_splits[windless_lightgbm_splits["split_feature"] == "wind_ms"]
print("LightGBM's missing types on wind_ms:", on_wind["missing_type"].unique().tolist())

windless_xgboost_splits = windless_xgboost.get_booster().trees_to_dataframe()
on_wind = windless_xgboost_splits[windless_xgboost_splits["Feature"] == "wind_ms"]
print("XGBoost's wind_ms splits:", len(on_wind), "| sending gaps to No:", (on_wind["Missing"] == on_wind["No"]).sum())

LightGBM's missing types on wind_ms: ['None']
XGBoost's wind_ms splits: 819 | sending gaps to No: 819


Neither model learned anything about a missing wind, and both predict without a warning. LightGBM's
wind splits have the missing type `None`, and it reads the gap as 0 m/s, a calm night, which cools the
most. XGBoost's wind splits all send a gap to the `No` side, the stronger winds, so a gap reads as the
strongest wind in the training rows, 26.5 m/s. LightGBM's mean prediction falls from −0.986 to
−2.574 °C and XGBoost's rises from −0.953 to −0.052, and the validation loss is about three times what
it was with the wind for LightGBM, 7.788 against 2.553, and about twice for XGBoost, 5.521 against
2.544. Compare the columns with gaps in new data against the columns that had gaps in training, and
find the missing readings or set those rows aside before predicting.

In [19]:
trained_with_gaps = set(X_train.columns[X_train.isna().any()])
new_gaps = set(no_wind.columns[no_wind.isna().any()])
print("columns with gaps the models were never trained on:", sorted(new_gaps - trained_with_gaps))

columns with gaps the models were never trained on: ['wind_ms']


### No error, and missing answers that LightGBM trains on as 0 °C

A tenth of the training answers are lost, and their rows are left in the table.

In [20]:
lost = y_train.copy()
lost.iloc[np.random.default_rng(8).choice(len(lost), 1500, replace=False)] = np.nan
settings = dict(n_estimators=150, learning_rate=0.1, random_state=0, verbose=-1)

clean = lgb.LGBMRegressor(**settings).fit(X_train, y_train)
with_lost = lgb.LGBMRegressor(**settings).fit(X_train, lost)
as_zero = lgb.LGBMRegressor(**settings).fit(X_train, lost.fillna(0.0))

print("same predictions as answers of 0 °C:", np.array_equal(with_lost.predict(X_valid), as_zero.predict(X_valid)))
print("validation loss, every answer:", round(mean_squared_error(y_valid, clean.predict(X_valid)), 3),
      "| 1,500 lost:", round(mean_squared_error(y_valid, with_lost.predict(X_valid)), 3))

same predictions as answers of 0 °C: True
validation loss, every answer: 2.562 | 1,500 lost: 3.164


LightGBM trains without an error, and its predictions are those of a model told that every lost day
had a minimum of 0 °C. The validation loss rises from 2.562 to 3.164. LightGBM's scikit-learn class
checks the answers only for features given as an array, where it raises `ValueError: Input y contains
NaN.`, so a DataFrame lets them through. The cell below gives it the numeric columns as an array.

In [21]:
try:
    lgb.LGBMRegressor(**settings).fit(X_train.drop(columns="terrain").to_numpy(), lost)
except ValueError as error:
    print(f"{type(error).__name__}: {error}")

ValueError: Input y contains NaN.


A row with no answer has nothing to teach, so drop it before training, which here gives a validation
loss of 2.543 on 13,500 rows, within 0.02 of the 2.562 with every answer.

In [22]:
known = lost.notna()
dropped = lgb.LGBMRegressor(**settings).fit(X_train[known], lost[known])
print("rows kept:", known.sum(), "| validation loss:", round(mean_squared_error(y_valid, dropped.predict(X_valid)), 3))

rows kept: 13500 | validation loss: 2.543


### `XGBoostError: Check failed: valid: Label contains NaN, infinity or a value too large.`

XGBoost meets the same mistake here with three answers lost. Its message starts with a time and the path
of the file in its own build, so the cell catches it and prints the message from the sentence after the
path.

In [23]:
import re


def xgboost_message(error):
    first_line = str(error).splitlines()[0]
    return "XGBoostError: " + re.sub(r"^\[[0-9:]+\] \S+:[0-9]+: ", "", first_line)


answers = y_train.copy()
answers.iloc[:3] = np.nan
try:
    xgb.XGBRegressor(n_estimators=50).fit(X_train, answers)
except xgb.core.XGBoostError as error:
    print(xgboost_message(error))

XGBoostError: Check failed: valid: Label contains NaN, infinity or a value too large.


XGBoost refuses to train on an answer that is `NaN`, and says so before it builds a tree. The fix is the
same as for LightGBM, keeping only the rows with an answer.

In [24]:
known = answers.notna()
model = xgb.XGBRegressor(n_estimators=50).fit(X_train[known], answers[known])
print("trained on", known.sum(), "rows")

trained on 14997 rows


### `` XGBoostError: Check failed: valid: Input data contains `inf` or a value too large, while `missing` is not set to `inf` ``

Snow depths are skewed, and taking their logarithm is a habit carried over from linear models. The log
of 0 cm is minus infinity.

In [25]:
import warnings

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always", RuntimeWarning)
    logged = X_train.assign(snow_cm=np.log(X_train["snow_cm"]))
for warning in caught:
    print(f"{warning.category.__name__}: {warning.message}")

try:
    xgb.XGBRegressor(n_estimators=50).fit(logged, y_train)
except xgb.core.XGBoostError as error:
    print("XGBoostError:", str(error).split(": ", 1)[1].splitlines()[0])

XGBoostError: Check failed: valid: Input data contains `inf` or a value too large, while `missing` is not set to `inf`


NumPy warns as it takes the log of 0, and XGBoost then stops at the minus infinity in the 5,853 rows
of bare ground, since it reads an infinity as a value too large unless `missing` is set to that same
infinity, here `-np.inf`. `np.log1p`, the log of one plus the depth, keeps 0 cm at 0. A tree splits on
the order of the values, though, and a log keeps their order, so the transform changes nothing for
these trees, as the cell below shows.

In [26]:
log1p_snow = lambda X: X.assign(snow_cm=np.log1p(X["snow_cm"]))
settings = dict(n_estimators=100, learning_rate=0.1, random_state=0)
raw_model = xgb.XGBRegressor(**settings).fit(X_train, y_train)
log_model = xgb.XGBRegressor(**settings).fit(log1p_snow(X_train), y_train)
print("same predictions with log1p as without:",
      np.array_equal(raw_model.predict(X_valid), log_model.predict(log1p_snow(X_valid))))

same predictions with log1p as without: True


## Recap

- You can count the gaps in a column, station by station, and say what a fill would put in them.
- You can read where a split sends the gaps, in LightGBM's `missing_direction` and XGBoost's `Missing`,
  and check the choice by hand.
- You can compare the gaps left in with a fill, a marker declared with `missing`, and no column at all.
- You can find a gap that a model was never trained on, and a lost answer, before either reaches a model.

## What is next

The **Categorical Features** notebook turns to the `terrain` column, and compares each library's own
handling of a category with one-hot columns.

---

&#8592; **Previous:** [Regularization and Sampling](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/07-regularization-and-sampling.ipynb)  &nbsp;·&nbsp;  [LightGBM and XGBoost, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Categorical Features](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/09-categorical-features.ipynb) &#8594;
